### 1. Dependencies & Data Inspection

In [ ]:
import kagglehub

# Download the latest version
path = kagglehub.dataset_download("vaukaofworlds/thecycloneimagedataset")

print("Path to dataset files:", path)
import os
print("Files in directory:", os.listdir(path))

100%|██████████| 1.54G/1.54G [00:15<00:00, 104MB/s]

Extracting files...


In [ ]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.utils.data import DataLoader
import torchvision
import torchvision.models as models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os

# Using the path confirmed by the kagglehub download
data_dir = '/root/.cache/kagglehub/datasets/vaukaofworlds/thecycloneimagedataset/versions/3'
h5_path = os.path.join(data_dir, 'Cyclone_Images.h5')
label_path = os.path.join(data_dir, 'Cyclone_Labels h5.npy')

# Loading the dataset
with h5py.File(h5_path, 'r') as f:
    # Use 'Images' key as per dataset structure
    # Only take the first channel (IR1) to match the 1-channel ResNet adjustment
    X = f['Images'][:, :, :, 0:1]

# Load labels from the .npy file enabling pickle for object arrays
y_raw = np.load(label_path, allow_pickle=True)

# Based on the error 'axis 1 with size 8', we inspect columns.
# Common TCIR structure: [Basin, No, Lat, Lon, Time, Vmax, ...]
# Looking at the raw data sample, index 6 appears to be the numeric Vmax field.
y = y_raw[:, 6].astype('float32')

# Convert NaNs in images to 0.0 and ensure float32
X = np.nan_to_num(X.astype('float32'), nan=0.0)

print(f"Images shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Sample Label (Vmax): {y[0]}")

### 2. Data Preprocessing & PyTorch Dataset Pipeline

In [ ]:
# Normalize to [0, 1]
X = X.astype('float32') / 255.0

# Train/Test Split (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

class CycloneDataset(data.Dataset):
    def __init__(self, images, labels):
        # Transpose to [N, C, H, W] for PyTorch
        self.images = torch.tensor(images).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels).float()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = CycloneDataset(X_train, y_train)
val_dataset = CycloneDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### 3. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_resnet18(num_channels=1):
    # Reverting to ResNet18 as requested
    model = models.resnet18(weights=None)

    # Modify first layer for single channel IR images
    model.conv1 = nn.Conv2d(num_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

    # Final regression head
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(device)

model = get_resnet18(num_channels=1)
print(f"ResNet18 loaded to {device}")

### 4. Training & Validation Loop

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(X_train):.4f} | Val MSE: {val_loss/len(X_val):.4f}")

### 5. IMD Classification & Sample Visualizer

In [ ]:
def get_imd_category(wind_speed):
    if wind_speed < 17: return "Low Pressure Area"
    elif 17 <= wind_speed <= 27: return "Depression"
    elif 28 <= wind_speed <= 33: return "Deep Depression"
    elif 34 <= wind_speed <= 47: return "Cyclonic Storm"
    elif 48 <= wind_speed <= 63: return "Severe Cyclonic Storm"
    elif 64 <= wind_speed <= 89: return "Very Severe Cyclonic Storm"
    elif 90 <= wind_speed <= 119: return "Extremely Severe Cyclonic Storm"
    else: return "Super Cyclonic Storm"

# Inference visualization
model.eval()
imgs, labels = next(iter(val_loader))
with torch.no_grad():
    preds = model(imgs.to(device)).cpu().numpy().flatten()

plt.figure(figsize=(12, 4))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(imgs[i][0], cmap='gray')
    actual_cat = get_imd_category(labels[i].item())
    pred_cat = get_imd_category(preds[i])
    plt.title(f"Act: {labels[i].item():.1f} ({actual_cat})\nPred: {preds[i]:.1f} ({pred_cat})")
    plt.axis('off')
plt.tight_layout()
plt.show()

### 6. Saving and Loading the Model
To deploy the model, you first need to save the trained weights. In PyTorch, we save the `state_dict`.

In [ ]:
# Save the model weights
torch.save(model.state_dict(), 'cyclone_resnet18.pth')
print("Model saved as cyclone_resnet18.pth")

# To 'deploy' or use it later:
def load_trained_model(weights_path):
    # Re-initialize the architecture as ResNet18
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, 1)

    # Load weights
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    return model.to(device)

# Example usage
deployed_model = load_trained_model('cyclone_resnet18.pth')
print("Model successfully reloaded for inference.")

### Deployment Strategy
1. **Web API**: Use **FastAPI** to create an endpoint where users can upload a cyclone image and receive the predicted intensity.
2. **Edge/Mobile**: Convert the model to **ONNX** or **TorchScript** to run it on devices with limited resources.
3. **Cloud**: Use services like Google Vertex AI or AWS SageMaker to host the model as a scalable web service.